In [1]:
import os
from google.colab import drive

# Hubungkan Google Colab ke Google Drive Anda
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import cv2
import os
import glob
import shutil

def remove_noise_dataset(base_dir, output_dir):
    # Cari semua subfolder di dalam direktori utama (train, valid, test)
    splits = ['train', 'valid', 'test']

    for split in splits:
        img_dir = os.path.join(base_dir, split, 'images')
        lbl_dir = os.path.join(base_dir, split, 'labels')

        # Cek apakah folder ada
        if not os.path.exists(img_dir) or not os.path.exists(lbl_dir):
            print(f"[-] Folder {split} tidak lengkap atau dilewati.")
            continue

        print(f"\n[+] Memproses noise reduction: {split.upper()}")

        # Buat folder output untuk hasil pembersihan noise
        out_img_dir = os.path.join(output_dir, split, 'images')
        out_lbl_dir = os.path.join(output_dir, split, 'labels')
        os.makedirs(out_img_dir, exist_ok=True)
        os.makedirs(out_lbl_dir, exist_ok=True)

        # Ambil semua file gambar
        img_extensions = ['*.jpg', '*.jpeg', '*.png', '*.JPG', '*.JPEG', '*.PNG']
        image_paths = []
        for ext in img_extensions:
            image_paths.extend(glob.glob(os.path.join(img_dir, ext)))

        for img_path in image_paths:
            filename = os.path.basename(img_path)
            basename, _ = os.path.splitext(filename)
            lbl_path = os.path.join(lbl_dir, f"{basename}.txt")

            # 1. Baca Gambar grayscale
            img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
            if img is None:
                continue

            # 2. Lakukan Noise Reduction menggunakan Gaussian Blur
            # Ksize (5, 5) adalah ukuran kernel. Harus bernilai ganjil.
            # Semakin besar angkanya, semakin halus/blur gambarnya.
            clean_img = cv2.GaussianBlur(img, (3, 3), 0)

            # Simpan gambar yang sudah bersih ke folder baru
            cv2.imwrite(os.path.join(out_img_dir, filename), clean_img)

            # 3. Salin File Label secara langsung (tidak ada perubahan koordinat)
            out_lbl_path = os.path.join(out_lbl_dir, f"{basename}.txt")
            if os.path.exists(lbl_path):
                shutil.copy(lbl_path, out_lbl_path)
            else:
                open(out_lbl_path, 'w').close()

        print(f"[v] Selesai membersihkan noise pada {len(image_paths)} gambar di {split}")

# --- KONFIGURASI JALUR FOLDER ---
# Mengambil input dari folder grayscale sebelumnya
INPUT_DATASET_DIR = "/content/drive/MyDrive/SIB/preprocessing/dataset-grayscaling"
OUTPUT_DATASET_DIR = "/content/drive/MyDrive/SIB/preprocessing/dataset-cleaning-noise"

# Jalankan fungsi noise reduction
remove_noise_dataset(
    base_dir=INPUT_DATASET_DIR,
    output_dir=OUTPUT_DATASET_DIR
)
print("\n[SUKSES] Noise reduction selesai! Struktur dataset baru tersimpan di 'dataset_clean'.")


[+] Memproses noise reduction: TRAIN
[v] Selesai membersihkan noise pada 282 gambar di train

[+] Memproses noise reduction: VALID
[v] Selesai membersihkan noise pada 81 gambar di valid

[+] Memproses noise reduction: TEST
[v] Selesai membersihkan noise pada 40 gambar di test

[SUKSES] Noise reduction selesai! Struktur dataset baru tersimpan di 'dataset_clean'.
